In [1]:
import torch

## Tensors of different ranks

In [ ]:
def print_tensor_properties(x, name=""):
    print(f"\n##### {name}")
    print(f"Tensor: {x}")
    print(f"Shape: {tuple(x.shape)}")
    print(f"Dim: {x.dim()}")
    print(f"Numel: {x.numel()}") # total number of elements
    print(f"dtype: {x.dtype}")
    print(f"device: {x.device}")
    print(f"requires_grad: {x.requires_grad}")

print_tensor_properties(torch.rand(()), "SCALAR (0D)")
print_tensor_properties(torch.rand(2), "VECTOR (1D)")
print_tensor_properties(torch.rand(2,2), "MATRIX (2D)")
print_tensor_properties(torch.rand(2,2,2), "3D TENSOR")
print_tensor_properties(torch.rand(2,2,2,2), "4D TENSOR")


##### SCALAR (0D)
Tensor: 0.48747164011001587
Shape: ()
Dim: 0
Numel: 1
dtype: torch.float32
device: cpu
requires_grad: False

##### VECTOR (1D)
Tensor: tensor([0.1371, 0.0118])
Shape: (2,)
Dim: 1
Numel: 2
dtype: torch.float32
device: cpu
requires_grad: False

##### MATRIX (2D)
Tensor: tensor([[0.2530, 0.2487],
        [0.5429, 0.2653]])
Shape: (2, 2)
Dim: 2
Numel: 4
dtype: torch.float32
device: cpu
requires_grad: False

##### 3D TENSOR
Tensor: tensor([[[0.8878, 0.7362],
         [0.5800, 0.0567]],

        [[0.8694, 0.3724],
         [0.6297, 0.8568]]])
Shape: (2, 2, 2)
Dim: 3
Numel: 8
dtype: torch.float32
device: cpu
requires_grad: False

##### 4D TENSOR
Tensor: tensor([[[[0.7819, 0.8529],
          [0.0550, 0.3344]],

         [[0.4293, 0.2752],
          [0.8861, 0.5349]]],


        [[[0.3373, 0.8029],
          [0.0145, 0.3688]],

         [[0.3967, 0.2534],
          [0.8918, 0.2121]]]])
Shape: (2, 2, 2, 2)
Dim: 4
Numel: 16
dtype: torch.float32
device: cpu
requires_grad: False


## Memory layout intuition: stride, storage_offset, transpose effects

In [32]:
A = torch.arange(12).reshape(3, 4)
print("A:\n", A)
print("A.shape:", A.shape)
print("A.stride():", A.stride())
print("A.storage_offset():", A.storage_offset())
print("A contiguous?", A.is_contiguous())

B = A.t()
print("\nB = A.t():\n", B)
print("B.shape:", B.shape)
print("B.stride():", B.stride())
print("B.storage_offset():", B.storage_offset())
print("B contiguous?", B.is_contiguous())

# Show they share the same underlying storage (same data pointer)
# (storage() is OK for learning; untyped_storage() is the newer API)
print("Same storage pointer?", A.storage().data_ptr() == B.storage().data_ptr())

A:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
A.shape: torch.Size([3, 4])
A.stride(): (4, 1)
A.storage_offset(): 0
A contiguous? True

B = A.t():
 tensor([[ 0,  4,  8],
        [ 1,  5,  9],
        [ 2,  6, 10],
        [ 3,  7, 11]])
B.shape: torch.Size([4, 3])
B.stride(): (1, 4)
B.storage_offset(): 0
B contiguous? False
Same storage pointer? True


## View on contiguous tensor

In [34]:
A = torch.arange(12).reshape(3, 4)
print("A:\n", A)
print("A.stride():", A.stride(), "| contiguous?", A.is_contiguous())

V = A.view(2, 6)
print("\nV = A.view(2,6):\n", V)
print("V.stride():", V.stride(), "| contiguous?", V.is_contiguous())
print("A and V share storage?", A.storage().data_ptr() == V.storage().data_ptr())

# Prove it is a view by modifying V and observing A change
V[0, 0] = -999
print("\nAfter V[0,0] = -999:")
print("A:\n", A)
print("V:\n", V)

A:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
A.stride(): (4, 1) | contiguous? True

V = A.view(2,6):
 tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])
V.stride(): (6, 1) | contiguous? True
A and V share storage? True

After V[0,0] = -999:
A:
 tensor([[-999,    1,    2,    3],
        [   4,    5,    6,    7],
        [   8,    9,   10,   11]])
V:
 tensor([[-999,    1,    2,    3,    4,    5],
        [   6,    7,    8,    9,   10,   11]])


## View on non-contiguous tensor

Fix with contiguous()

In [35]:
A = torch.arange(12).reshape(3, 4)
B = A.t()  # non-contiguous
print("A contiguous?", A.is_contiguous())
print("B = A.t() contiguous?", B.is_contiguous())
print("B.stride():", B.stride())

# view should fail
try:
    C = B.view(12)
    print("C (view) succeeded unexpectedly:", C)
except RuntimeError as e:
    print("view failed as expected:")
    print("  ", e)

# Fix 1: make contiguous then view
B_contig = B.contiguous()
C1 = B_contig.view(12)
print("\nFix 1: B.contiguous().view(12)")
print("B_contig contiguous?", B_contig.is_contiguous())
print("C1:", C1, "| contiguous?", C1.is_contiguous())

# Fix 2: reshape (will return view if possible, else copy)
C2 = B.reshape(12)
print("\nFix 2: B.reshape(12)")
print("C2:", C2, "| contiguous?", C2.is_contiguous())
print("B and C2 share storage?", B.storage().data_ptr() == C2.storage().data_ptr())



A contiguous? True
B = A.t() contiguous? False
B.stride(): (1, 4)
view failed as expected:
   view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

Fix 1: B.contiguous().view(12)
B_contig contiguous? True
C1: tensor([ 0,  4,  8,  1,  5,  9,  2,  6, 10,  3,  7, 11]) | contiguous? True

Fix 2: B.reshape(12)
C2: tensor([ 0,  4,  8,  1,  5,  9,  2,  6, 10,  3,  7, 11]) | contiguous? True
B and C2 share storage? False


## Permute dimensions + prove index mapping + strides/contiguity

In [37]:
X = torch.randn(2, 3, 4)
print("X.shape:", X.shape)
print("X.stride():", X.stride(), "| contiguous?", X.is_contiguous())

X1 = X.permute(2, 0, 1)  # (2,3,4) -> (4,2,3)
print("\nX1 = X.permute(2,0,1)")
print("X1.shape:", X1.shape)
print("X1.stride():", X1.stride(), "| contiguous?", X1.is_contiguous())

X2 = X.permute(1, 2, 0)  # (2,3,4) -> (3,4,2)
print("\nX2 = X.permute(1,2,0)")
print("X2.shape:", X2.shape)
print("X2.stride():", X2.stride(), "| contiguous?", X2.is_contiguous())

# Prove values moved as expected by checking index mapping
i, j, k = 1, 2, 3
print("\nIndex mapping check:")
print("X[i,j,k] =", X[i, j, k].item())
print("X1[k,i,j] =", X1[k, i, j].item(), " (should match X[i,j,k])")
print("X2[j,k,i] =", X2[j, k, i].item(), " (should match X[i,j,k])")

X.shape: torch.Size([2, 3, 4])
X.stride(): (12, 4, 1) | contiguous? True

X1 = X.permute(2,0,1)
X1.shape: torch.Size([4, 2, 3])
X1.stride(): (1, 12, 4) | contiguous? False

X2 = X.permute(1,2,0)
X2.shape: torch.Size([3, 4, 2])
X2.stride(): (4, 1, 12) | contiguous? False

Index mapping check:
X[i,j,k] = 0.38270965218544006
X1[k,i,j] = 0.38270965218544006  (should match X[i,j,k])
X2[j,k,i] = 0.38270965218544006  (should match X[i,j,k])


## Simulate CNN tensor layout

In [ ]:
images = torch.randn(8, 3, 32, 32)  # NCHW

## Manual broadcasting reasoning

In [ ]:
A = torch.randn(3,4)
b = torch.randn(4)

## Broadcasting failure case

## Complex broadcasting

## Implement matmul manually

## Naive softmax

## Numerically stable softmax

## Basic gradient

## Gradient through softmax